 ### Student name: Kristyan Connolly

 ### Student ID: 25245295

 

In [292]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle

from sklearn import manifold #needed for multidimensional scaling (MDS) and t-SNE
from sklearn import cluster #needed for k-Means clustering
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, FunctionTransformer, OneHotEncoder, OrdinalEncoder #needed for data preparation

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn import set_config

from sklearn.impute import SimpleImputer


Task 1: Data Preparation Pipeline
Open a new Jupyter notebook and name it etivity2.ipynb. In this notebook, create a data preparation pipeline that applies the same kind of transformations that you applied as part of e-tivity 1. It is OK to leave some of the transformations outside the pipeline but aim at including as many transformations as you can within the pipeline. Follow the notebook Tutorial 2 - Clustering and Manifold Learning.ipynb as an example.

In [293]:
# First lets load the data into a dataframe
df = pd.read_csv('./bank.csv')

df_original = df.copy()

In [294]:
# Print first 5 rows of the dataframe
df.head(5)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,subscribed
0,32.0,technician,single,tertiary,no,392,yes,no,cellular,1,apr,957,2,131,2,failure,no
1,39.0,technician,divorced,secondary,no,688,yes,yes,cellular,1,apr,233,2,133,1,failure,no
2,59.0,retired,married,secondary,no,1035,yes,yes,cellular,1,apr,126,2,239,1,failure,no
3,47.0,blue-collar,married,secondary,no,398,yes,yes,cellular,1,apr,274,1,238,2,failure,no
4,54.0,retired,married,secondary,no,1004,yes,no,cellular,1,apr,479,1,307,1,failure,no


In [295]:
# Frequency counts for categorical attributes
categorical_attributes  = df.select_dtypes(include=['str']).columns

print("\nCategorical attributes:\n")
print(list(categorical_attributes))

print("\nFrequency counts for categorical attributes:\n")

for column in categorical_attributes: 
    print("\n")
    print(df[column].value_counts(dropna=False))
    print("--------------------------------")


Categorical attributes:

['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'subscribed']

Frequency counts for categorical attributes:



job
management       461
technician       348
blue-collar      298
admin.           247
services         165
retired          162
student           96
unemployed        69
self-employed     64
entrepreneur      45
housemaid         35
NaN               10
Name: count, dtype: int64
--------------------------------


marital
married     1111
single       674
divorced     215
Name: count, dtype: int64
--------------------------------


education
secondary    995
tertiary     684
primary      217
NaN          104
Name: count, dtype: int64
--------------------------------


default
no     1985
yes      15
Name: count, dtype: int64
--------------------------------


housing
no     1037
yes     963
Name: count, dtype: int64
--------------------------------


loan
no     1750
yes     250
Name: count, dtype: int64


In [296]:

# shift and log the balance column to avoid log(0)
# np.log(df['balance'] + abs(min(df['balance'])) + 1)
def log_shift(X):
    """log(x + |min(x)| + 1) per column — same idea as np.log(df['balance'] + abs(min(df['balance'])) + 1)."""
    X = np.asarray(X, dtype=float)
    min_x = np.min(X, axis=0, keepdims=True)
    # avoid log(0)
    return np.log(X + np.abs(min_x) + 1)


# generic log transformer
log_transformer = FunctionTransformer(log_shift)

# Pipeline: robust scale then log on the balance column
balance_pipeline = Pipeline(
    steps=[
        ("scaler", RobustScaler()),
        ("distribution_transform", log_transformer)
    ]
)

# Pipeline: standard scale then impute missing values with median
age_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("imputer", SimpleImputer(strategy='median', missing_values=np.nan))
    ]
)

education_order = [
    "primary", 
    "secondary", 
    "tertiary"
]

education_encoder = OrdinalEncoder(categories=[education_order], handle_unknown='use_encoded_value', unknown_value=np.nan)

education_encoder.fit_transform(df[['education']])

print(education_encoder.categories_)

education_pipeline = Pipeline(
    steps=[
        ("encoder", education_encoder)
    ]
)
########################################################

month_order = [
    "jan", 
    "feb", 
    "mar", 
    "apr", 
    "may", 
    "jun", 
    "jul",
    "aug", 
    "sep", 
    "oct", 
    "nov", 
    "dec"
]

month_encoder = OrdinalEncoder(categories=[month_order])

month_encoder.fit_transform(df[['month']])

print(month_encoder.categories_)

month_pipeline = Pipeline(
    steps=[
        ("encoder", month_encoder)
    ]
)


########################################################

one_hot_encoder = OneHotEncoder()



job_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)


contact_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)

marital_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)

housing_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)

poutcome_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)

default_pipeline = Pipeline(
    steps=[
        ("encoder", one_hot_encoder)
    ]
)


# Next, we use ColumnTransformer to construct the main preprocessing pipeline which consists of all data preprocessing steps applied to all columns.
# the aim is to replace the transformation from e-tivity 1 with a pipeline

preprocess_pipeline = ColumnTransformer(
  [("balance", balance_pipeline, ["balance"]),
   ("age", age_pipeline, ["age"]),
   ("education", education_pipeline, ["education"]),
   ("month", month_pipeline, ["month"]),
   ("job", job_pipeline, ["job"]),
   ("contact", contact_pipeline, ["contact"]),
   ("marital", marital_pipeline, ["marital"]),
   ("housing", housing_pipeline, ["housing"]),
   ("poutcome", poutcome_pipeline, ["poutcome"]),
   ("default", default_pipeline, ["default"])]
)


transformed = preprocess_pipeline.fit_transform(df)
 

#print("df_transformed class: ", type(transformed))

df_transformed = pd.DataFrame(transformed)

preprocess_pipeline


[array(['primary', 'secondary', 'tertiary'], dtype=object)]
[array(['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep',
       'oct', 'nov', 'dec'], dtype=object)]


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('balance', ...), ('age', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name`

In [297]:
df_transformed.head()

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
0,0.668238,-0.766677,2.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.768220,-0.216413,1.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
2,0.873966,1.355771,1.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
3,0.670367,0.412461,1.0,3.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0
4,0.864961,0.962725,1.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0


In [298]:
df_original.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,subscribed
0,32.0,technician,single,tertiary,no,392,yes,no,cellular,1,apr,957,2,131,2,failure,no
1,39.0,technician,divorced,secondary,no,688,yes,yes,cellular,1,apr,233,2,133,1,failure,no
2,59.0,retired,married,secondary,no,1035,yes,yes,cellular,1,apr,126,2,239,1,failure,no
3,47.0,blue-collar,married,secondary,no,398,yes,yes,cellular,1,apr,274,1,238,2,failure,no
4,54.0,retired,married,secondary,no,1004,yes,no,cellular,1,apr,479,1,307,1,failure,no
